# Synthea → OMOP → calibration → validation

This notebook demonstrates the full data-driven workflow of the Hospital Digital Twin Simulator:

1. **Load** OMOP data (real Synthea export, or a self-contained synthetic fallback).
2. **Calibrate** inter-service transitions and length-of-stay from the OMOP visits.
3. **Validate** the model's exponential length-of-stay assumption with a Kolmogorov–Smirnov goodness-of-fit test.
4. **Simulate** a calibrated scenario and plot occupancy.

## Using real Synthea data

[Synthea](https://github.com/synthetichealth/synthea) generates synthetic patients exportable to the OMOP CDM:

```bash
java -jar synthea-with-dependencies.jar -p 1000 --exporter.omop.export=true
```

Put the resulting OMOP CSVs (`person.csv`, `condition_occurrence.csv`, `visit_occurrence.csv`, `procedure_occurrence.csv`) in a folder and pass its path to `load_or_generate(...)` below. With no path, a synthetic *Synthea-like* dataset with known ground truth is generated so the notebook runs offline.

In [ ]:
import sys
from pathlib import Path

# Make the tested example module importable from the notebooks/ directory.
sys.path.insert(0, str(Path.cwd().parent / "examples"))
import synthea_omop_etl as etl

## 1–2. Load OMOP and calibrate

Pass a directory of OMOP CSVs to use real data; `None` generates the synthetic fallback.

In [ ]:
OMOP_PATH = None  # e.g. "../data/synthea_omop"

dataset, synthetic = etl.load_or_generate(OMOP_PATH)
patients, routing, mean_los, stays = etl.calibrate(dataset)

print(f"source: {'synthetic' if synthetic else OMOP_PATH}")
print(f"patients: {len(patients)} | stays: {len(stays)}")
print("estimated ED routing:", routing["ED"])
print("estimated mean LOS (days):", {k: round(v, 2) for k, v in mean_los.items()})

### Ground-truth recovery (synthetic data only)

With the synthetic fallback we know the true transition probabilities, so we can check that calibration recovers them.

In [ ]:
if synthetic:
    for dest, truth in sorted(etl.GROUND_TRUTH_TRANSITIONS["ED"].items()):
        print(f"{dest:<10} true={truth:.2f}  estimated={routing['ED'].get(dest, 0):.2f}")

## 3. Validate the length-of-stay assumption

The simulation engine samples length of stay from an **exponential** distribution. Here we test
that assumption against the observed durations with a one-sample Kolmogorov–Smirnov test
(`p > 0.05` means the exponential fit is not rejected). With real data, a rejection would
motivate a heavier-tailed distribution — a documented direction for future work.

In [ ]:
report = etl.validate_length_of_stay(stays, mean_los)
for service, r in report.items():
    verdict = "OK" if r["exponential_ok"] else "REJECTED"
    print(f"{service:<6} n={r['n']:>4}  mean_LOS={r['observed_mean']:>5}d  "
          f"D={r['ks_D']:.3f}  p={r['p_value']:.3f}  -> exponential {verdict}")

## 4. Simulate a calibrated scenario and plot occupancy

Requires the `viz` extra (`pip install 'hospital_simulator[viz]'`).

In [ ]:
from hospital_simulator import Scenario, run_scenario, run_replications
from hospital_simulator.plotting import plot_occupancy

scenario = Scenario(
    name="pneumonia_calibrated", days=90, warmup_days=15,
    arrival_rate_per_day=12.0,
    service_capacities={"ED": 30, "ICU": 16, "Ward": 60},
    routing=routing, mean_los_days=mean_los, seed=2026,
)

rep = run_replications(scenario, 40)
print(rep.render_summary(metrics=["blocked_transfers", "ICU.saturation_days", "deaths"]))

plot_occupancy(run_scenario(scenario))